# Part 10: Benchmark and Cap Sensitivity

This notebook runs the reproducible Part 10 Python runner in Colab. Part 10 extends the frozen Part 1-9 evidence chain with matched fixed BTC, cap-only, risk-cap sensitivity, and fine-grid sensitivity diagnostics. It does not overwrite Part 1-9 outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/project_edhec_paper')
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

assert (PROJECT_DIR / 'data_2026' / 'cleaned').exists(), 'Cleaned input folder not found.'
assert (PROJECT_DIR / 'experiments' / 'part10_benchmark_cap_sensitivity' / 'run_part10_benchmark_cap_sensitivity.py').exists(), 'Part 10 runner not found.'

In [ ]:
!python -m pip install -q -r experiments/part10_benchmark_cap_sensitivity/requirements-part10.txt

## Path configuration

Each run directory first tries the canonical Drive path, then the downloaded zip folder structure with `*_outputs/...`.

In [ ]:
from pathlib import Path

def choose_existing(*candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(candidates[0])

INPUT_DIR = Path('data_2026/cleaned')
PART1_RUN_DIR = choose_existing('outputs/part1_btc_macro_state/colab_part1_seed42', 'outputs/part1_btc_macro_state_outputs/part1_btc_macro_state/colab_part1_seed42')
PART2_RUN_DIR = choose_existing('outputs/part2_portfolio_risk_budget/colab_part2_seed42', 'outputs/part2_portfolio_risk_budget_outputs/part2_portfolio_risk_budget/colab_part2_seed42')
PART4_RUN_DIR = choose_existing('outputs/part4_conditional_btc_allocation/colab_part4_seed42', 'outputs/part4_conditional_btc_allocation_outputs/part4_conditional_btc_allocation/colab_part4_seed42')
OUTPUT_DIR = Path('outputs/part10_benchmark_cap_sensitivity_outputs/part10_benchmark_cap_sensitivity')
RUN_ID = 'colab_part10_seed42'

print('INPUT_DIR:', INPUT_DIR)
print('PART1_RUN_DIR:', PART1_RUN_DIR)
print('PART2_RUN_DIR:', PART2_RUN_DIR)
print('PART4_RUN_DIR:', PART4_RUN_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
!python experiments/part10_benchmark_cap_sensitivity/run_part10_benchmark_cap_sensitivity.py \
  --input-dir "{INPUT_DIR}" \
  --part1-run-dir "{PART1_RUN_DIR}" \
  --part2-run-dir "{PART2_RUN_DIR}" \
  --part4-run-dir "{PART4_RUN_DIR}" \
  --output-dir "{OUTPUT_DIR}" \
  --run-id "{RUN_ID}" \
  --seed 42

In [ ]:
import json
import pandas as pd

RUN_DIR = OUTPUT_DIR / RUN_ID
RESULTS = RUN_DIR / 'results'
FIGURES = RUN_DIR / 'figures'

validation = json.loads((RESULTS / 'output_validation_summary.json').read_text())
print(json.dumps(validation, indent=2))
assert validation['status'] == 'passed'

In [ ]:
display(pd.read_csv(RESULTS / 'part10_target_weight_performance_summary.csv').head(12))
display(pd.read_csv(RESULTS / 'part10_pairwise_benchmark_comparison.csv').head(16))
display(pd.read_csv(RESULTS / 'part10_executed_state_weights.csv').query("risk_budget_cap == 0.10"))

In [ ]:
from IPython.display import Image, display

for name in [
    'part10_cap_sensitivity_btc_weight.png',
    'part10_cap_sensitivity_risk_contribution.png',
    'part10_conditional_vs_matched_fixed_performance.png',
    'part10_conditional_vs_cap_only_performance.png',
]:
    print(name)
    display(Image(filename=str(FIGURES / name)))

In [ ]:
import shutil

zip_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print('Created zip:', zip_path)